In [ ]:
# Settings + Data fetching
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png r::r-raster")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")
system("conda install conda-forge::r-fnn")
system("conda install conda-forge::r-cluster bioconda::r-bioregion")
system("conda install -y conda-forge::r-hrbrthemes conda-forge::r-viridis")


In [ ]:
library(R.utils)
library(tidyverse) # because who can live without the tidyverse?
library(dplyr)
library(tidyr)
library(utils)

library(ncdf4)
library(lubridate)
library(RColorBrewer)
library(lattice)

library(cluster)
library(data.table)
library(bioregion)
library(FNN)

library(terra)     
library(sf)        
library(ggplot2)

library(viridis)
library(hrbrthemes) 


In [ ]:
# SET DIRECTORIES
workdir <- getwd()
dataDir <- paste(workdir,"Data",sep = "/")
outputDir <- paste(workdir,"outputs/collection",sep = "/")
scriptDir <- paste(workdir,"scripts",sep = "/")

In [ ]:
# GENERAL FUNCTIONS

# ==============================================================================
# CREATE REPERTORY
# ==============================================================================
create.directory <- function(name.directory){
    ifelse(!dir.exists(file.path(name.directory)),
        dir.create(file.path(name.directory)),
        "Directory Exists")
}

# ==============================================================================
# DOWNLOAD FILENAME
# ==============================================================================
download.filename <- function(filename, url){
    options(timeout = 600)  # 10 minutes
    
    if(file.exists(filename)){
        cat(filename, "is (are) already in your repertory.")
    } else {
        download.file(url, filename, mode = "wb")
        print('File Downloaded')
    }
}

# ==============================================================================
# UNZIP FILENAME
# ==============================================================================
unzip.file <- function(filename, type = "gz"){
    filename.length <- nchar(filename)
    print(filename, filename.length)

    start <- 1

    if(type == "gz"){
        # Case Gunzip
        end <- filename.length-3
    }
    else{
        # Case Unzip
        end <- filename.length-4
    }

    unzip.filename <- substr(filename,start, end)
    print(unzip.filename)

    
    # Case gunzip
    if(!file.exists(unzip.filename)){
        if(type == "gz"){
            R.utils::gunzip(filename, overwrite=FALSE, remove=TRUE, BFR.SIZE=1e+07)
            }
        else{
            utils::unzip(filename, overwrite=FALSE )
            }
        cat(filename, "successfully unzipped!")
    }

    return(unzip.filename)
}

# ==============================================================================
# EXPORT AS A CSV 
# ==============================================================================
export.csv <- function(directory = outputDir, table.output, filename){
    table_filename <- paste(directory,filename, sep="/")
    write.table(table.output, table_filename, row.names=TRUE, col.names = TRUE, sep=",")
    print(paste(filename,"output exported.", sep = " "))
    }

# ==============================================================================
# MOVE TO DATA DIRECTORY 
# ==============================================================================
move.file <- function(filename, new.path){
    new.filename <- paste(new.path, filename, sep = "/")
    file.rename(from=filename, to=new.filename)
    return(new.filename)
}


In [ ]:
# Read CSV file
# Example CSV structure:
wind_data <- rbind(c(-75,40,3.2,1.5), c(-74,41,2.8,0.9)) %>% as.data.frame
colnames(wind_data) <- c("lon","lat","u","v")
wind_data %>% head
# wind_data <- read.csv("wind_vectors.csv", stringsAsFactors = FALSE)

# Validate data
if (!all(c("lon", "lat", "u", "v") %in% names(wind_data))) {
  stop("CSV must contain columns: lon, lat, u, v")
}

# Quick plot of wind vectors
ggplot(wind_data, aes(x = lon, y = lat)) +
  geom_segment(aes(xend = lon + u, yend = lat + v),
               arrow = arrow(length = unit(0.2, "cm")),
               color = "blue") +
  coord_fixed() +
  theme_minimal() +
  labs(title = "Wind Vectors from CSV",
       x = "Longitude", y = "Latitude")


In [ ]:
# ==============================================================================
# WIND FILENAMES + URLS
# ==============================================================================
create.directory(dataDir)
windspeed.urls <- c("https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis/Monthlies/surface/wspd.mon.mean.nc",
                   "https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis/Monthlies/surface/wspd.sig995.mon.1981-2010.ltm.nc",
                   "https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis/Monthlies/surface/wspd.sig995.mon.ltm.1991-2020.nc")

windspeed.files <- c("wspd.mon.mean.nc",
                     "wspd.sig995.mon.ltm.1981-2010.nc",
                     "wspd.sig995.mon.ltm.1991-2020.nc")


vwind.urls <- c("https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis/Monthlies/surface/vwnd.sig995.mon.mean.nc",
                "https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis/Monthlies/surface/vwnd.sig995.day.ltm.1981-2010.nc",
                "https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis/Monthlies/surface/vwnd.sig995.day.ltm.1991-2020.nc")

vwind.files <- c("vwnd.sig995.mon.mean.nc",
                 "vwnd.sig995.day.ltm.1981-2010.nc",
                 "vwnd.sig995.day.ltm.1991-2020.nc")


uwind.urls <- c("https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis/Monthlies/surface/uwnd.sig995.mon.mean.nc",
                "https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis/Monthlies/surface/uwnd.sig995.mon.ltm.1981-2010.nc",
                "https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis/Monthlies/surface/uwnd.sig995.mon.ltm.1991-2020.nc")

uwind.files <- c("uwnd.sig995.mon.mean.nc",
                 "uwnd.sig995.day.ltm.1981-2010.nc",
                 "uwnd.sig995.day.ltm.1991-2020.nc")


In [ ]:
i <- 1
download.filename(windspeed.files[i], windspeed.urls[i]) ; windspeed <- move.file(windspeed.files[i], dataDir)
download.filename(vwind.files[i], vwind.urls[i]) ; vwind <- move.file(vwind.files[i], dataDir)
download.filename(uwind.files[i], uwind.urls[i]) ; uwind <- move.file(uwind.files[i], dataDir)


In [ ]:
# Load NetCDF files
u <- rast(uwind)
v <- rast(vwind)
speed <- rast(windspeed)  # optional (can compute instead)

# If needed: compute speed yourself
# speed <- sqrt(u^2 + v^2)

# Convert to dataframe
df_u <- as.data.frame(u, xy = TRUE)
df_v <- as.data.frame(v, xy = TRUE)
df_s <- as.data.frame(speed, xy = TRUE)

# Merge all
df <- df_u %>%
  rename(u = 3) %>%
  left_join(df_v %>% rename(v = 3), by = c("x", "y")) %>%
  left_join(df_s %>% rename(speed = 3), by = c("x", "y"))

# Optional: thin arrows (important for readability)
df_arrow <- df %>% 
  slice(seq(1, n(), by = 20))

colors <- length(unique(speed))
# Plot
ggplot(df_arrow) +
  geom_segment(aes(x = x, y = y,
                   xend = x + u * 0.5,
                   yend = y + v * 0.5,
                   color = speed),
               arrow = arrow(length = unit(0.1, "cm")),
               size = 0.4) +

  scale_color_gradientn(
    colours = c("dark blue", "blue", "cyan", "green", "yellow", "orange", "red", "dark red"),
    name = "Wind speed"
  ) +
  coord_quickmap() +
  theme_minimal()

In [ ]:
nc <- nc_open(windspeed)
print(nc)
nc_close(nc)

In [ ]:
urls_mpapd <- c(
  paste0("https://raw.githubusercontent.com/ccamlr/data/main/geographical_data/mpapd/CCAMLR_MPAPD_EPSG4326.",
         c("shp", "shx", "dbf", "prj")),
  paste0("https://raw.githubusercontent.com/ccamlr/data/main/geographical_data/mpapd/CCAMLR_MPAPD_EPSG6932.",
         c("shp", "shx", "dbf", "prj"))
)

lapply(urls_mpapd, function(u) {
  download.file(
    url = u,
    destfile = file.path(dataDir, basename(u)),
    mode = "wb"
  )
})

MPAPDs <- st_read(file.path(dataDir, "CCAMLR_MPAPD_EPSG4326.shp"))

# Reproject to the correct CRS --- 
if(crs(MPAPDs) != 4326){
    MPAPDs <- st_transform(MPAPDs, crs = 4326)
}


In [ ]:
# Load NetCDF files
u <- rast(uwind)
v <- rast(vwind)
speed <- rast(windspeed)  # optional (can compute instead)

# Read shapefile (EPSG:4326 version is easiest)
mpa <- vect(file.path(dataDir, "CCAMLR_MPAPD_EPSG4326.shp"))


## Option A — Mask the raster (keep only inside polygons)

In [ ]:
crs(u) <-"EPSG:4326"
crs(v) <- crs(speed) <- crs(u)

# --------------------------------------------------
# Define target region
# 30–150°E, 70–60°S
# --------------------------------------------------
ext_crop <- ext(30, 150, -70, -60)

# --------------------------------------------------
# Make sure MPA is in the same CRS as the raster
# --------------------------------------------------
mpa <- project(mpa, crs(u))

# --------------------------------------------------
# Crop first
# --------------------------------------------------
u_crop <- crop(u, ext_crop)
v_crop <- crop(v, ext_crop)
speed_crop <- crop(speed, ext_crop)

# --------------------------------------------------
# Mask
# --------------------------------------------------
u_masked <- mask(u_crop, mpa)
v_masked <- mask(v_crop, mpa)
speed_masked <- mask(speed_crop, mpa)

# --------------------------------------------------
# Convert to data frames
# --------------------------------------------------
df_u <- as.data.frame(u_masked, xy = TRUE, na.rm = TRUE)
df_v <- as.data.frame(v_masked, xy = TRUE, na.rm = TRUE)
df_s <- as.data.frame(speed_masked, xy = TRUE, na.rm = TRUE)

In [ ]:
print(u)
print(ext(u))
print(crs(u))
print("# --------------------------------------------------")
print(mpa)
print(ext(mpa))
print(crs(mpa))

In [ ]:
# Load NetCDF files
u <- rast(uwind)
v <- rast(vwind)
speed <- rast(windspeed)

# Correct the spatial extent of the NCEP 144 x 73 grid
ext(u) <- ext(0, 360, -90, 90)
ext(v) <- ext(0, 360, -90, 90)
ext(speed) <- ext(0, 360, -90, 90)

# Set geographic CRS
crs(u) <- "EPSG:4326"
crs(v) <- "EPSG:4326"
crs(speed) <- "EPSG:4326"

# Read MPA shapefile
mpa <- vect(file.path(dataDir, "CCAMLR_MPAPD_EPSG4326.shp"))

# Make sure MPA has same CRS
mpa <- project(mpa, crs(u))

# Desired region:
# 30–150°E, 70–60°S
ext_crop <- ext(30, 150, -70, -60)

# Crop first
u_crop <- crop(u, ext_crop)
v_crop <- crop(v, ext_crop)
speed_crop <- crop(speed, ext_crop)

# Then mask by MPA
u_masked <- mask(u_crop, mpa)
v_masked <- mask(v_crop, mpa)
speed_masked <- mask(speed_crop, mpa)

# Convert to data frames
df_u <- as.data.frame(u_masked, xy = TRUE, na.rm = TRUE)
df_v <- as.data.frame(v_masked, xy = TRUE, na.rm = TRUE)
df_s <- as.data.frame(speed_masked, xy = TRUE, na.rm = TRUE)

In [ ]:
crs(u) <-"EPSG:4326"
crs(v) <- crs(speed) <- crs(u)

##################################
# Merge u and v into one dataframe
##################################
df <- merge(df_u, df_v, by = c("x", "y"))

# Rename for clarity (adjust depending on layer names)
colnames(df) <- c("lon", "lat", "u", "v")

##################################
# Thin the vectors (VERY important)
##################################
df_thin <- df[seq(1, nrow(df), by = 20), ]

##################################
# Plot Antarctica + MPAs + wind vectors
##################################
df_thin <- df[seq(1, nrow(df), by = 20), ]

mpa_sf <- st_as_sf(mpa)

ggplot() +
  geom_sf(data = mpa_sf, fill = "lightblue", color = "black") +

  geom_segment(data = df_thin,
               aes(x = lon, y = lat,
                   xend = lon + u * 0.5,
                   yend = lat + v * 0.5),
               arrow = arrow(length = unit(0.1, "cm")),
               alpha = 0.7) +

  coord_sf(xlim = c(-180, 180), ylim = c(-90, -60)) +
  theme_minimal()

In [ ]:
crs(u) <-"EPSG:4326"
crs(v) <- crs(speed) <- crs(u)

##################################
# Merge u and v
##################################
df <- merge(df_u, df_v, by = c("x", "y"))
colnames(df) <- c("lon", "lat", "u", "v")

##################################
# Thin vectors
##################################
df_thin <- df[seq(1, nrow(df), by = 20), ]

##################################
# Convert MPA to sf
##################################
mpa_sf <- st_as_sf(mpa)

##################################
# Plot
##################################

ggplot() +
  # MPA polygons -- background
  geom_sf(
    data = mpa_sf,
    fill = "lightblue",
    color = "black",
    alpha = 0.3
  ) +
  # Wind arrows -- FOREGROUND
  geom_segment(
    data = df_thin,
    aes(
      x = lon,
      y = lat,
      xend = lon + u * 0.5,
      yend = lat + v * 0.5),
    color = "black",
    linewidth = 0.5,
    alpha = 1,
    arrow = arrow(
      length = unit(0.15, "cm"),
      type = "closed")) +
  coord_sf(
    xlim = c(30, 150),
    ylim = c(-70, -60),
    expand = FALSE) +
  theme_minimal()

In [ ]:
ggplot() +
  geom_raster(data = df_s, aes(x = x, y = y, fill = speed)) +
  geom_sf(data = mpa_sf, fill = NA, colour = "black") +
  geom_segment(
    data = df_wind,
    aes(
      x = x, y = y,
      xend = x + u * scale,
      yend = y + v * scale
    ),
    arrow = arrow(length = unit(0.15, "cm")),
    linewidth = 0.4
  )

## Option B — Convert raster → points and spatial join (most flexible)

In [ ]:
crs(u) <-"EPSG:4326"
crs(v) <- crs(speed) <- crs(u)

# Convert raster to points
pts <- as.points(u)  # or speed
pts_sf <- st_as_sf(pts)

sf_use_s2(FALSE)
# Convert MPA to sf
mpa_sf <- st_as_sf(mpa)
mpa_sf <- st_make_valid(mpa_sf)

# Transform points to match shapefile CRS
pts_sf <- st_transform(pts_sf, st_crs(mpa_sf))


In [ ]:
wind

In [ ]:
wind <- c(u, v, speed)
names(wind) <- c("u", "v", "speed")

df <- as.data.frame(wind, xy = TRUE, na.rm = TRUE)

# Subsample for arrows
df_arrow <- df %>%
  slice(seq(1, n(), by = 20))

In [ ]:
ggplot() +
  geom_sf(data = mpa_sf, fill = NA, color = "black", size = 0.3) +
  geom_segment(data = df_arrow,
               aes(x = x, y = y,
                   xend = x + u * 0.5,
                   yend = y + v * 0.5,
                   color = speed),
               arrow = arrow(length = unit(0.1, "cm")),
               size = 0.4) +
  scale_color_gradientn(
    colours = c("dark blue", "blue", "cyan", "green", "yellow", "orange", "red", "dark red"),
    name = "Wind speed"
  ) +
  coord_sf() +
  theme_minimal()

In [ ]:
df_u %>% head()
df_v %>% head()
df_s %>% head()

df %>% head()


In [ ]:
library(ncdf4)

# Open NetCDF file

nc <- nc_open(windspeed)

# Read variables (names depend on your dataset)
lon <- ncvar_get(nc, "lon")
lat <- ncvar_get(nc, "lat")
u <- ncvar_get(nc, "u10")  # U-component at 10m
v <- ncvar_get(nc, "v10")  # V-component at 10m

nc_close(nc)

# Flatten into a data frame
wind_df <- expand.grid(lon = lon, lat = lat)
wind_df$u <- as.vector(u)
wind_df$v <- as.vector(v)

# Plot
library(ggplot2)
ggplot(wind_df, aes(x = lon, y = lat)) +
  geom_segment(aes(xend = lon + u, yend = lat + v),
               arrow = arrow(length = unit(0.15, "cm")),
               color = "red") +
  coord_fixed() +
  theme_minimal() +
  labs(title = "Wind Vectors from NetCDF",
       x = "Longitude", y = "Latitude")


In [ ]:
wind_df$speed <- sqrt(wind_df$u^2 + wind_df$v^2)
wind_df$direction <- (atan2(wind_df$u, wind_df$v) * 180 / pi + 360) %% 360


https://psl.noaa.gov/data/gridded/data.ncep.reanalysis.html
* 

https://www.ncei.noaa.gov/products/international-comprehensive-ocean-atmosphere-data-set
* https://www.ncei.noaa.gov/data/international-comprehensive-ocean-atmosphere/v3/archive/nrt/monthly/


In [ ]:
# Load required libraries
library(ncdf4)         # To read netCDF files
library(ggplot2)       # For plotting
library(viridis)       # For color scales in the plot
library(sf)            # For handling spatial objects
library(rnaturalearth) # For retrieving coastline data
library(rnaturalearthdata)

In [ ]:
# Defined locations
netcdf_file <- "path/to/your/file.nc"       # Full path to the netCDF file
output_dir  <- "path/to/your/directory/"   # Output directory to save the map

In [ ]:
# Names of variables in the netCDF file (modify according to your file)
var_lon <- "longitude"  # Variable name for longitude
var_lat <- "latitude"   # Variable name for latitude
var_u   <- "uo"         # Variable name for the u component of current
var_v   <- "vo"         # Variable name for the v component of current

In [ ]:
# Open the netCDF file
nc_data <- nc_open(netcdf_file)

# Extract the data (variable names and dimensions may vary)
lon <- ncvar_get(nc_data, var_lon)
lat <- ncvar_get(nc_data, var_lat)
u   <- ncvar_get(nc_data, var_u)
v   <- ncvar_get(nc_data, var_v)

# Close the netCDF file
nc_close(nc_data)

In [ ]:
# Verify that dimensions match (assuming a grid of lon x lat)
if (length(lon) * length(lat) != length(u)) {
  stop("Variable dimensions do not match. Check your netCDF file.")
}

In [ ]:
# Create a grid with all combinations of lon and lat
grid <- expand.grid(lon = lon, lat = lat)

In [ ]:
# Assign the u and v values to the grid (convert to vectors)
grid$u <- as.vector(u)
grid$v <- as.vector(v)
grid$speed <- sqrt(grid$u^2 + grid$v^2)  # Calculate current speed

# Adjust 'skip' to modify the number of arrows plotted
skip <- 10
grid_subset <- grid[seq(1, nrow(grid), by = skip), ]

# Determine geographic extent for plotting
min_lon <- min(grid$lon, na.rm = TRUE)
max_lon <- max(grid$lon, na.rm = TRUE)
min_lat <- min(grid$lat, na.rm = TRUE)
max_lat <- max(grid$lat, na.rm = TRUE)

In [ ]:
## ---- Loading Coastline Data ----
# Load coastline data using rnaturalearth
world <- ne_countries(scale = "medium", returnclass = "sf")

In [ ]:
p <- ggplot() +
  geom_sf(data = world, fill = NA, color = "black", linewidth = 0.5) +
  geom_segment(
    data = grid_subset,
    aes(x = lon, y = lat, xend = lon + u, yend = lat + v, color = speed),
    # Adjust arrow size by modifying the length (e.g., unit(0.15, "cm") to another value)
    arrow = arrow(length = unit(0.15, "cm")),
    # Adjust arrow thickness by changing 'linewidth'
    linewidth = 0.5
  ) +
  # Change the color gradient by modifying the 'option' parameter in scale_color_viridis (e.g., "magma", "inferno", etc.)\n  scale_color_viridis(option = "plasma") +
  coord_sf(xlim = c(min_lon, max_lon), ylim = c(min_lat, max_lat), expand = FALSE) +
  theme_minimal() +
  labs(
    title = "Ocean Current Vectors (Subset)",  # Modify the title as needed
    x = "Longitude",
    y = "Latitude",
    color = "Speed (m/s)"
  )

print(p)

In [ ]:
output_file <- file.path(output_dir, "ocean_current_vectors.png")
ggsave(output_file, p, width = 15, height = 12, dpi = 300)